In [ ]:
import matplotlib.pyplot as plt

# 1. Define your base font size
base_size = 12

# 2. Update matplotlib's global configurations (rcParams)
plt.rcParams.update({
    # Font Sizes
    'font.size': base_size,             # Default text size
    'axes.titlesize': base_size + 2,     # Title of subplots
    'axes.labelsize': base_size,         # X and Y axis labels
    'xtick.labelsize': base_size - 2,    # X-axis tick labels
    'ytick.labelsize': base_size - 2,    # Y-axis tick labels
    'legend.fontsize': base_size - 1,    # Legend text size
    'figure.titlesize': base_size + 4,   # Figure main title
    
    # Figure properties (Optional but highly recommended for consistency)
    'figure.figsize': (10, 5),           # Default width and height of figures in inches
    'figure.dpi': 100,                   # Screen resolution/crispness of figures
    'savefig.dpi': 300,                  # Resolution when saving plots
    'axes.grid': True,                   # Enable grids by default
    'grid.alpha': 0.3,                   # Make the grid lines subtle and light
    'grid.linestyle': '--'               # Dashed grid lines
})

## Task 1

In [ ]:
# Gertrud

## Task 2

In [ ]:
# Gertrud

## Task 3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from numpy.linalg import inv, matrix_power

P = np.array([
    [0.9915, 0.005,  0.0025, 0,      0.001 ],
    [0,      0.986,  0.005,  0.004,  0.005 ],
    [0,      0,      0.992,  0.003,  0.005 ],
    [0,      0,      0,      0.991,  0.009 ],
    [0,      0,      0,      0,      1.0   ]
])

def simulate_woman(P):
    """Simulates a single woman's path until death (state 5, index 4)."""
    state = 0
    path = [state]
    while state != 4:
        state = np.random.choice(5, p=P[state])
        path.append(state)
    return path

np.random.seed(42)
n_sims = 1000
lifetimes = []
local_recurrence_count = 0
states_at_120 = []

for _ in range(n_sims):
    path = simulate_woman(P)
    lifetimes.append(len(path) - 1)
    
    if 1 in path or 3 in path:
        local_recurrence_count += 1
        
    # State at t = 120
    if len(path) - 1 >= 120:
        states_at_120.append(path[120])
    else:
        states_at_120.append(4) # Already absorbed in death state


# 1. Setup Phase-Type Matrices
Ps = P[:4, :4]
ps = P[:4, 4]
pi_s = np.array([1.0, 0.0, 0.0, 0.0])
I = np.eye(4)
N = inv(I - Ps)
e = np.ones(4)


mean_theory = pi_s @ N @ e

# Theoretical PMF and CDF
t_max = max(lifetimes)
t_vals = np.arange(1, t_max + 1)
pmf_theory = np.array([pi_s @ matrix_power(Ps, t-1) @ ps for t in t_vals])




mean_sim = np.mean(lifetimes)
std_sim = np.std(lifetimes)


bin_edges = np.linspace(1, t_max, 16)
obs_counts, _ = np.histogram(lifetimes, bins=bin_edges)

# Compute expected probabilities for each bin from theoretical PMF
exp_counts = []
for i in range(len(bin_edges) - 1):
    low, high = int(np.floor(bin_edges[i])), int(np.floor(bin_edges[i+1]))

    prob = np.sum(pmf_theory[(t_vals >= low) & (t_vals < high)])
    exp_counts.append(prob * n_sims)

exp_counts = np.array(exp_counts)

exp_counts = exp_counts * (np.sum(obs_counts) / np.sum(exp_counts))

chi2_stat_fit, p_val_fit = stats.chisquare(f_obs=obs_counts, f_exp=exp_counts)


print("="*60)
print(f"{'STATISTIC':<30} | {'SIMULATED':<12} | {'THEREOTICAL':<12}")
print("="*60)
print(f"{'Mean Lifetime (Months)':<30} | {mean_sim:<12.2f} | {mean_theory:<12.2f}")

print("-"*60)
print(f"Global Chi-Square Goodness-of-Fit Test (15 bins):")
print(f"  - Chi2 Statistic: {chi2_stat_fit:.4f}")
print(f"  - p-value: {p_val_fit:.4f} (A high p-value means the distributions match perfectly!)")
print("="*60)

# Prepare bin labels for the x-axis using the given scheme
bin_labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-2)] + [f"{int(bin_edges[-2])}+"]
x = np.arange(len(bin_labels))
width = 0.42

fig, ax = plt.subplots()

observed = obs_counts
expected = exp_counts

ax.bar(x - width/2, observed, width=width, color="#4C78A8", edgecolor="black", label="Observed")
ax.bar(x + width/2, expected, width=width, color="#F58518", edgecolor="black", alpha=0.85, label="Expected (theory)")

ax.set_title("Lifetimes", pad=10)
ax.set_xlabel("Months")
ax.set_ylabel("Number of women")
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=45, ha="center")
ax.legend()

plt.tight_layout()
plt.show()

## Task 4

In [ ]:
np.random.seed(42)

accepted_lifetimes = []
while len(accepted_lifetimes) < 1000:
    path = simulate_woman(P)
    
    # 1. Condition: Survived at least 12 months
    survived_12 = (len(path) - 1 >= 12)
    
    # 2. Condition: Cancer reappeared at any point in months 1 to 12 
    early_recurrence = any(state in [1, 2, 3] for state in path[1:13])
    
    if survived_12 and early_recurrence:
        accepted_lifetimes.append(len(path) - 1)

print(f"Task 4: Expected lifetime of a woman with early recurrence: {np.mean(accepted_lifetimes):.2f} months")

## Task 5

In [ ]:
np.random.seed(42)

n_iters = 100
n_women = 200

Y_vals, X_vals = [], []

for _ in range(n_iters):
    batch_lifetimes = [len(simulate_woman(P)) - 1 for _ in range(n_women)]
    # Y = fraction dying within 350 months (our target estimator)
    Y_vals.append(np.mean(np.array(batch_lifetimes) <= 350))
    # X = mean lifetime of the batch (our control variate)
    X_vals.append(np.mean(batch_lifetimes))

Y_vals = np.array(Y_vals)
X_vals = np.array(X_vals)

# Compute optimal control coefficient c*
cov_matrix = np.cov(X_vals, Y_vals)
c_star = -cov_matrix[0, 1] / cov_matrix[0, 0]

# Controlled estimate
# Y_c = Y + c* * (X - E[X])
Y_c = Y_vals + c_star * (X_vals - mean_theory)

var_crude = np.var(Y_vals, ddof=1)
var_cv = np.var(Y_c, ddof=1)
reduction = (1 - (var_cv / var_crude)) * 100

print(f"Task 5: Crude MC estimate P(T <= 350): {np.mean(Y_vals):.4f} (Var: {var_crude:.6f})")
print(f"Task 5: Control Variate estimate:      {np.mean(Y_c):.4f} (Var: {var_cv:.6f})")
print(f"Task 5: Variance reduction achieved:   {reduction:.2f}%")

## Task 7

In [ ]:
# Gertrud

## Task 8

In [ ]:
# Gertrud

## Task 9

## Task 11

## Task 12

## Task 13